# 02 — The Value object
Lecture 1, ~14-18% (search: "class value takes a single scalar"). Phase 1, item 2 in [PLAN](../knowledge/PLAN.md). Note: [02-value-object](../knowledge/notes/02-value-object.md). Prereq: [python-dunder-methods](../knowledge/prereqs/python-dunder-methods.md).

**Where this fits.** In notebook 01, every slope cost one full re-run of the expression. To beat that, the expression has to remember how it was built. Today: a number that keeps a receipt.

**Done when** `d = a*b + c` gives `Value(data=4.0)`, and from `d` alone you can find out what operation made it and which values went in.

Pure Python. No numpy, no torch.

## 1. Wrap a number
A class `Value` that takes one number and stores it as `.data`. Give it a `__repr__` so `Value(2.0)` displays as `Value(data=2.0)`.

Before adding `__repr__`, look at what the notebook shows for a `Value`. Then add it.

In [36]:
class Value:
    def __init__(self, data):
        self.data = data
        # this field will be used to track the previous Value in the computational graph
        self._prev = ()
        # this field will be used to track the operation that produced this Value
        self._op = ''

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        # create a new Value object to represent the result of the addition
        out = Value(self.data + other.data)
        # set the _prev field to track the previous Values in the computational graph
        out._prev = (self, other)
        # set the _op field to indicate that this Value was produced by addition
        out._op = '+'
        return out


    def __mul__(self, other):
        # create a new Value object to represent the result of the multiplication
        out = Value(self.data * other.data)
        # set the _prev field to track the previous Values in the computational graph
        out._prev = (self, other)
        # set the _op field to indicate that this Value was produced by multiplication
        out._op = '*'
        return out

## 2. Add two Values
`a = Value(2.0)`, `b = Value(-3.0)`. Try `a + b` **before** writing anything, and read the error.

Then add `__add__`. It returns a **new** Value. Expect `Value(data=-1.0)`.

Question: inside `__add__`, is `self.data + other.data` a Value plus a Value, or a float plus a float?

In [37]:
# a = Value(2.0)
# b = Value(-3.0)
# print(a + b)  # Unsupported operand type(s) for +: 'Value' and 'Value'

a = Value(2.0)
b = Value(-3.0)
print(a + b)  # Value(data=-1.0)

Value(data=-1.0)


## 3. Multiply, and rebuild d
Add `__mul__`. Then `c = Value(10.0)` and `d = a*b + c`. Expect `Value(data=4.0)`, the same `d` as notebook 01 section 5.

Also call the dunders by hand, e.g. `a.__mul__(b).__add__(c)`, and check you get the same thing.

In [38]:
c = Value(10.0)
d = a * b + c
print(d)  # Value(data=4.0)
# print(a.__mul__(b).__add__(c)) # Value(data=4.0)

Value(data=4.0)


## 4. Keep the receipt: children
Give `Value` a way to remember which Values made it. Karpathy passes a tuple `_children` (empty by default) into the constructor and stores it as a set in `self._prev`.

Now `d._prev` should contain the -6.0 and the 10.0. What should `a._prev` be?

In [39]:
print(a._prev) # ()
print(b._prev) # ()
print(c._prev) # ()
print(d._prev) # (Value(data=-6.0), Value(data=10.0))

()
()
()
(Value(data=-6.0), Value(data=10.0))


## 5. Keep the receipt: the operation
Add `_op`: an empty string for values you create by hand, `'+'` or `'*'` for values made by an operation.

Then, starting from `d` only, print the whole recipe: d's op and children, then the op and children of the -6.0. Don't use `a`, `b` or `c` by name.

In [ ]:
def traverse_graph(value):
    for node in value._prev:
        print(node, node._op)
        traverse_graph(node)

# d = a * b + c
traverse_graph(d)

Value(data=-6.0) *
Value(data=2.0) 
Value(data=-3.0) 
Value(data=10.0) 


## 6. Break it
Predict each one, then run it.

1. `a + 1`. What happens, and why? (Look at which line fails.)
2. `1 + a`. Same error or different? Which object's dunder does Python try first?
3. `e = a * a`. How many children does `e._prev` have? Should it be 1 or 2? (Hint: `_prev` is a set.) Keep this in mind for item 6.
4. `x = Value(2.0); x.data = 5.0` after building `y = x * b`. Does `y.data` change? What does that tell you about when the math happens?

In [41]:
# print(a + 1) # AttributeError: 'int' object has no attribute 'data'
# print(1 + a) # TypeError: unsupported operand type(s) for +: 'int' and 'Value'  (a different error)
e = a * a
print(e._prev) # (Value(data=2.0), Value(data=2.0))

# x = Value(2.0); y = x * b; x.data = 5.0  → y.data stays -6.0
# The math happens once, when y is created.

# OPEN (next session):
#  1. a + 1: which line of __add__ fails, and why does it name 'int'?
#  2. 1 + a: whose __add__ does Python call first? Why a different error?
#  3. a * a: should the receipt have one entry or two?
#  4. After the update step changes .data, what must happen before the next backward pass?

(Value(data=2.0), Value(data=2.0))


## Notes / findings
Your own words: what does `Value` store, and why can't a plain float do the same job?

In [42]:
a = Value(2.0)
for child in a._prev:
    print(child)